# 설명만으로 도구 선택하기

공개 이름·스키마·동작을 유지하고 description만 바꿉니다.

## 준비

In [1]:
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = Path("data/orders.csv")
assert data_path.exists(), "노트북은 labs/part5/ch19-model-tools에서 실행합니다."
print("Python 실습 준비 완료")

Python 실습 준비 완료


In [2]:
import os
from langchain_openai import ChatOpenAI
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
model = ChatOpenAI(model=MODEL, temperature=0)
print({"model": MODEL, "temperature": 0, "예상_절별_비용_USD_상한": 1.0})

{'model': 'gpt-5.4-mini', 'temperature': 0, '예상_절별_비용_USD_상한': 1.0}


In [3]:
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")

## 모호한 설명

In [4]:
from langchain.tools import tool
@tool("list_orders_by_status", description="주문 정보를 찾습니다.")
def bad_status(status: str) -> list[dict]:
    return fetch_orders(data_path, status=status)
@tool("get_order_by_id", description="주문 정보를 찾습니다.")
def bad_id(order_id: str) -> dict:
    rows = fetch_orders(data_path)
    return next((r for r in rows if r["order_id"] == order_id), {"error": "주문을 찾지 못했습니다."})
@tool("summarize_order_amounts", description="주문 정보를 찾습니다.")
def bad_summary(status: str) -> dict:
    rows = fetch_orders(data_path, status=status)
    return {"status": status, "count": len(rows), "total": sum(r["amount"] for r in rows)}

## 구체적인 설명

In [5]:
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)

In [6]:
@tool
def get_order_by_id(order_id: str) -> dict:
    """정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다."""
    rows = fetch_orders(data_path)
    return next((r for r in rows if r["order_id"] == order_id), {"error": "주문을 찾지 못했습니다."})

@tool
def summarize_order_amounts(status: str) -> dict:
    """주문 상태별 건수와 총 주문 금액을 집계합니다."""
    rows = fetch_orders(data_path, status=status)
    return {"status": status, "count": len(rows), "total": sum(r["amount"] for r in rows)}

## 조건 확인

In [7]:
bad_set = [bad_status, bad_id, bad_summary]
good_set = [list_orders_by_status, get_order_by_id, summarize_order_amounts]
for bad, good in zip(bad_set, good_set):
    print({"same_name": bad.name == good.name, "same_args": bad.args == good.args,
           "descriptions": [bad.description, good.description]})

{'same_name': True, 'same_args': True, 'descriptions': ['주문 정보를 찾습니다.', '주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다.']}
{'same_name': True, 'same_args': True, 'descriptions': ['주문 정보를 찾습니다.', '정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다.']}
{'same_name': True, 'same_args': True, 'descriptions': ['주문 정보를 찾습니다.', '주문 상태별 건수와 총 주문 금액을 집계합니다.']}


## 반복 조건

In [8]:
questions = [("paid 주문 목록을 보여 줘", "list_orders_by_status"),
             ("O103 주문을 찾아 줘", "get_order_by_id"),
             ("paid 주문의 건수와 총액을 계산해 줘", "summarize_order_amounts")]
REPEATS = 3
print({"questions": len(questions), "repeats": REPEATS, "calls_per_condition": 9})

{'questions': 3, 'repeats': 3, 'calls_per_condition': 9}


## 전체 호출 로그

In [9]:
def run_selection(tools):
    bound = model.bind_tools(tools, parallel_tool_calls=False)
    logs = []
    for question, expected in questions:
        for repeat in range(REPEATS):
            response = bound.invoke(question)
            logs.append({"question": question, "expected": expected, "repeat": repeat + 1,
                         "tool_calls": response.tool_calls, "usage": response.usage_metadata})
    return logs
bad_logs = run_selection(bad_set)
good_logs = run_selection(good_set)
for condition, logs in [("모호", bad_logs), ("구체", good_logs)]:
    for row in logs:
        print(condition, row)

모호 {'question': 'paid 주문 목록을 보여 줘', 'expected': 'list_orders_by_status', 'repeat': 1, 'tool_calls': [{'name': 'list_orders_by_status', 'args': {'status': 'paid'}, 'id': 'call_2QBhUghzme7uXRjROpSa1z8U', 'type': 'tool_call'}], 'usage': {'input_tokens': 175, 'output_tokens': 19, 'total_tokens': 194, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
모호 {'question': 'paid 주문 목록을 보여 줘', 'expected': 'list_orders_by_status', 'repeat': 2, 'tool_calls': [{'name': 'list_orders_by_status', 'args': {'status': 'paid'}, 'id': 'call_44AiK0DRQHjZg0avlDjTiZNt', 'type': 'tool_call'}], 'usage': {'input_tokens': 175, 'output_tokens': 19, 'total_tokens': 194, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
모호 {'question': 'paid 주문 목록을 보여 줘', 'expected': 'list_orders_by_status', 'repeat': 3, 'tool_calls': [{'name': 'list_orders_by_status', 'args': {'status': 'paid'}, 'id': 'call_WMpGVuFC8Ey

## 수동 채점

In [10]:
def grade(logs):
    names = [r["tool_calls"][0]["name"] if len(r["tool_calls"]) == 1 else None for r in logs]
    return {"correct": sum(n == r["expected"] for n, r in zip(names, logs)), "total": len(logs)}
print({"ambiguous": grade(bad_logs), "specific": grade(good_logs)})

{'ambiguous': {'correct': 9, 'total': 9}, 'specific': {'correct': 9, 'total': 9}}


## 실험 결과 해석

질문 3개와 조건별 9회에서 관찰한 결과만 해석합니다.